# Q1: Filter the data for the country assigned to you.

> Path note: this notebook now looks for files relative to the repository. In Colab, clone or upload the repository before running the code.


In [ ]:
import pandas as pd
from pathlib import Path


def locate_first(*relative_path_options):
    current = Path.cwd().resolve()
    search_roots = [current, *current.parents]
    for parts in relative_path_options:
        for base in search_roots:
            candidate = base.joinpath(*parts)
            if candidate.exists():
                return candidate
    options = ", ".join("/".join(parts) for parts in relative_path_options)
    raise FileNotFoundError(
        f"Could not locate any of these files: {options}. Please place the WDI CSV file in the repository Data folder first."
    )


OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


df = pd.read_csv(
    locate_first(
        ("archive/legacy_wdi_materials", "Data", "WDICSV.csv"),
        ("Data", "WDICSV.csv"),
    )
)


In [ ]:
### Example: keep the observations for the United States.
df_US = df[df["Country Name"] == "United States"]


# Q2: How many rows and columns does that country-level dataset have?

In [ ]:
df_US.shape


# Q3: Reshape your country-level data into panel format.

In [ ]:
### Convert the country-level data to panel form.
data_pd = df_US.drop(columns="Indicator Code").melt(
    id_vars=["Country Name", "Country Code", "Indicator Name"],
    var_name="Year",
).pivot_table(
    values="value",
    index=["Country Name", "Country Code", "Year"],
    columns="Indicator Name",
).reset_index().rename_axis("", axis=1)

data_pd.head()


In [ ]:
### Recheck data types.
data_pd.dtypes

### Convert `Year` from string to integer.
data_pd["Year"] = data_pd["Year"].astype(str).astype(int)
data_pd.dtypes


# Q4: Export the panel dataset.

In [ ]:
data_pd.to_csv(OUTPUT_DIR / "WDI_US.csv", index=False)


# Q5: How many missing values does each variable contain?

In [ ]:
isna_data = data_pd.isna().sum().sort_values(ascending=True)
isna_data


# Bonus
# Q6: Select the ten indicators with the fewest missing values and reorganize them into a panel dataset.

In [ ]:
### This is one possible solution for reference.
top_10_features = isna_data.head(10).index
panel_data = data_pd.set_index(["Country Name", "Year"]).loc[:, top_10_features].reset_index().reset_index()
panel_data
